# Season OPR and cOPR

This notebook loads a season match CSV such as `frc_matches_2026.csv`, computes season-long OPR for every team, and computes cOPRs for every numeric `score_breakdown` component found in the file.

It should work (not tested) for any year-specific match export as long as the CSV follows the same schema.

In [ ]:
import ast
import re
from pathlib import Path

import numpy as np
import pandas as pd


In [ ]:
def resolve_input_csv() -> Path:
    raw = input("Enter match CSV path or season year [auto-detect latest frc_matches_*.csv]: ").strip()
    if not raw:
        candidates = sorted(
            Path.cwd().glob("frc_matches_*.csv"),
            key=lambda path: path.stat().st_mtime,
            reverse=True,
        )
        if not candidates:
            raise FileNotFoundError("No frc_matches_*.csv file was found in the current directory.")
        return candidates[0]
    if raw.isdigit():
        return Path(f"frc_matches_{raw}.csv")
    return Path(raw)


def parse_team_keys(value) -> list[str]:
    if isinstance(value, list):
        return [str(item) for item in value]
    if value is None or pd.isna(value):
        return []
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        try:
            parsed = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return [value]
        if isinstance(parsed, list):
            return [str(item) for item in parsed]
    return []


def team_number_from_key(team_key: str) -> int:
    if isinstance(team_key, str):
        match = re.search(r"(\d+)", team_key)
        if match:
            return int(match.group(1))
    return int(team_key)


def numeric_value(value) -> float:
    if isinstance(value, (int, float, np.integer, np.floating)) and not pd.isna(value):
        return float(value)
    coerced = pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]
    return float(coerced) if pd.notna(coerced) else 0.0


def infer_components(df: pd.DataFrame) -> list[str]:
    components: list[str] = []
    seen: set[str] = set()
    for column in df.columns:
        if not column.startswith("score_breakdown."):
            continue
        parts = column.split(".", 2)
        if len(parts) != 3:
            continue
        component = parts[2]
        if component in seen:
            continue
        series = pd.to_numeric(df[column], errors="coerce")
        if series.notna().any():
            seen.add(component)
            components.append(component)
    return components


In [ ]:
csv_path = resolve_input_csv()
if not csv_path.exists():
    raise FileNotFoundError(f"Could not find input CSV: {csv_path}")

df = pd.read_csv(csv_path, low_memory=False)
components = infer_components(df)

completed_matches = []
team_numbers = set()

for _, row in df.iterrows():
    red_score = pd.to_numeric(pd.Series([row.get("alliances.red.score")]), errors="coerce").iloc[0]
    blue_score = pd.to_numeric(pd.Series([row.get("alliances.blue.score")]), errors="coerce").iloc[0]
    if pd.isna(red_score) or pd.isna(blue_score):
        continue

    red_team_keys = parse_team_keys(row.get("alliances.red.team_keys"))
    blue_team_keys = parse_team_keys(row.get("alliances.blue.team_keys"))
    if not red_team_keys or not blue_team_keys:
        continue

    row_dict = row.to_dict()
    completed_matches.append(row_dict)
    team_numbers.update(team_number_from_key(team_key) for team_key in red_team_keys)
    team_numbers.update(team_number_from_key(team_key) for team_key in blue_team_keys)

if not completed_matches:
    raise RuntimeError(f"No completed matches were found in {csv_path}.")

team_numbers = sorted(team_numbers)
team_index = {team_number: index for index, team_number in enumerate(team_numbers)}

rows = []
total_scores = []
component_vectors = {component: [] for component in components}
match_appearances = {team_number: 0 for team_number in team_numbers}

for match in completed_matches:
    for color in ("red", "blue"):
        team_keys = parse_team_keys(match.get(f"alliances.{color}.team_keys"))
        if not team_keys:
            continue

        row_vector = np.zeros(len(team_numbers), dtype=float)
        for team_key in team_keys:
            team_number = team_number_from_key(team_key)
            row_vector[team_index[team_number]] = 1.0
            match_appearances[team_number] += 1

        rows.append(row_vector)
        total_scores.append(numeric_value(match.get(f"alliances.{color}.score")))

        for component in components:
            component_vectors[component].append(
                numeric_value(match.get(f"score_breakdown.{color}.{component}"))
            )

design_matrix = np.vstack(rows)
target_columns = [np.asarray(total_scores, dtype=float)] + [
    np.asarray(component_vectors[component], dtype=float) for component in components
]
target_matrix = np.column_stack(target_columns)

solutions, *_ = np.linalg.lstsq(design_matrix, target_matrix, rcond=None)
opr_values = solutions[:, 0]
component_oprs = {
    component: solutions[:, index + 1] for index, component in enumerate(components)
}

result = pd.DataFrame(
    {
        "team_number": team_numbers,
        "team_key": [f"frc{team_number}" for team_number in team_numbers],
        "matches_played": [match_appearances[team_number] for team_number in team_numbers],
        "opr": opr_values,
    }
)

for component in components:
    safe_name = component.replace(".", "_")
    result[f"copr_{safe_name}"] = component_oprs[component]

result = result.sort_values("opr", ascending=False).reset_index(drop=True)

output_path = csv_path.with_name(f"{csv_path.stem}_opr.csv")
result.to_csv(output_path, index=False)

print(f"Loaded {len(completed_matches):,} completed matches from {csv_path.name}")
print(f"Computed OPR/cOPR for {len(result):,} teams")
print(f"Wrote {output_path}")
result.head(20)
